> ## 🚀 Initial Setup
>
> ---
>
> ### 1️⃣ Databricks Workspace
>
> You can connect to your **own Databricks workspace** using a SQL warehouse or all-purpose cluster.
>
> To connect, you'll need:
> - **Server hostname** — the hostname of your Databricks workspace.
> - **HTTP path** — the HTTP path of your SQL warehouse or cluster.
> - **Access token** — a personal access token generated from your Databricks user settings.

> **Optional:**

> - **Catalog** — the Unity Catalog catalog containing the TPC-H data.
> - **Schema** — the schema (database) containing the TPC-H tables.
>
> ---
>
> 💡 **Tip:**
> Store these credentials as environment variables for easy access and security:
> ```env
> DATABRICKS_HOST=<your-workspace-hostname>
> DATABRICKS_HTTP_PATH=<your-warehouse-or-cluster-http-path>
> DATABRICKS_TOKEN=<your-personal-access-token>
> ```

> ## 🔌 Installing the Databricks Connector
>
> Make sure to have the **`databricks-sql-connector`** package installed:
>
> ---
>
> - **If you're working inside the repo**:
> ```bash
> pip install -e ".[databricks]"
> ```
>
> - **Or install the connector directly**:
> ```bash
> pip install databricks-sql-connector
> ```

> ## Importing Required Libraries
>
> ---

In [ ]:
# Import libraries
import pydough
import datetime
import os

> ## 🔑 Loading Credentials and Connecting to Databricks
>
> ---
>
> ### 1️⃣ Load Credentials from Environment Variables
> - The following environment variables are read using `os.getenv(...)`:
>   - `DATABRICKS_HOST`
>   - `DATABRICKS_HTTP_PATH`
>   - `DATABRICKS_TOKEN`
>
> ---
>
> ### 2️⃣ Databricks-PyDough `connect_database()` Parameters
> - **`server_hostname`** **(required)**: Hostname of the Databricks workspace.
> - **`http_path`** **(required)**: HTTP path of the SQL warehouse or cluster.
> - **`access_token`** **(required)**: Personal access token used for authentication.
> - **`catalog`** *(optional)*: Unity Catalog catalog to use for the connection.
> - **`schema`** *(optional)*: Schema (database) to use for the connection.
>
> ---
>
> ### 3️⃣ Connect to Databricks Using PyDough
> - `pydough.active_session.load_metadata_graph(...)`
>   Loads a metadata graph mapping your Databricks schema (used for query planning/optimizations).
> - `connect_database(...)`
>   Uses the loaded credentials to establish a live connection to your Databricks workspace.
>
> ---
>
> **⚠️ Notes:**
> - Ensure the environment variables are set and contain **valid values**.
> - The metadata graph path must point to a **valid JSON file** representing your schema.

In [ ]:
# Set Databricks connection parameters
databricks_host: str = os.getenv("DATABRICKS_HOST")
databricks_http_path: str = os.getenv("DATABRICKS_HTTP_PATH")
databricks_token: str = os.getenv("DATABRICKS_TOKEN")
databricks_catalog: str = "tpch_s3_pq"
databricks_schema: str = "TPCH_SF1"

# Metadata setup
pydough.active_session.load_metadata_graph("../../tests/test_metadata/databricks_sample_graphs.json", "TPCH")

# Setup Databricks connection
pydough.active_session.connect_database("databricks",
    server_hostname=databricks_host,
    http_path=databricks_http_path,
    access_token=databricks_token,
    catalog=databricks_catalog,
    schema=databricks_schema,
)

> ## ✨ Enabling PyDough's Jupyter Magic Commands
>
> ---
>
> This step loads the **`pydough.jupyter_extensions`** module, which adds custom magic commands (like `%%pydough`) to your notebook.
>
> ---
>
> ### 📊 What These Magic Commands Do
> - **Write PyDough directly** in notebook cells using:
>   ```python
>   %%pydough
>   ```
> - **Automatically render** query results inside the notebook.
>
> ---
>
> ### 💻 How It Works
> This is a **Jupyter-specific feature** — the `%load_ext` command dynamically loads these extensions into your **current notebook session**:
> ```python
> %load_ext pydough.jupyter_extensions
> ```

In [ ]:
%load_ext pydough.jupyter_extensions

> ## 📊 Running TPC-H Query 1 with PyDough in Databricks
>
> ---
>
> This cell runs **TPC-H Query 1** using **PyDough**.
>
> ---
>
> ### 📝 What the Query Does
> - **Computes summary statistics**: sums, averages, and counts for orders.
> - **Groups by**: `return_flag` and `line_status`.
> - **Filters by**: a shipping date cutoff.
>
> ---
>
> ### 📤 Output
> - `pydough.to_df(output)` converts the result to a **Pandas DataFrame**.
> - This makes it easy to inspect and analyze results directly in Python.
>
> ---

In [ ]:
%%pydough
# TPCH Q1
output = (lines.WHERE((ship_date <= datetime.date(1998, 12, 1)))
        .PARTITION(name="groups", by=(return_flag, status))
        .CALCULATE(
            L_RETURNFLAG=return_flag,
            L_LINESTATUS=status,
            SUM_QTY=SUM(lines.quantity),
            SUM_BASE_PRICE=SUM(lines.extended_price),
            SUM_DISC_PRICE=SUM(lines.extended_price * (1 - lines.discount)),
            SUM_CHARGE=SUM(
                lines.extended_price * (1 - lines.discount) * (1 + lines.tax)
            ),
            AVG_QTY=AVG(lines.quantity),
            AVG_PRICE=AVG(lines.extended_price),
            AVG_DISC=AVG(lines.discount),
            COUNT_ORDER=COUNT(lines),
        )
        .ORDER_BY(L_RETURNFLAG.ASC(), L_LINESTATUS.ASC())
)

pydough.to_df(output)

> ## 🔍 Inspecting the Generated SQL
>
> ---
>
> `pydough.to_sql(output)` returns the SQL query (translated for the Databricks/Spark SQL dialect) that PyDough generates for the query above, without executing it.

In [ ]:
print(pydough.to_sql(output))